In [ ]:
import numpy as np
import scipy.integrate as integrate
import scipy.fft as fft
import matplotlib.pyplot as plt
import soundfile as sf

# Parâmetros do modelo linear do alto-falante
m = 14.35e-3  # Massa (kg)
b = 0.786     # Amortecimento (kg/s)
k = 1852      # Constante de mola (N/m)
Bl = 4.95     # Fator de força (N/A)
L = 266e-6    # Indutância (H)
R = 3.3       # Resistência (Ohm)

# Função para representar o modelo linear do alto-falante
def linear_model(t, y, Vin):
    i, x, v = y
    Vin_t = np.interp(t, time, Vin)  # Interpolar Vin para o tempo t
    di_dt = (Vin_t - R * i - Bl * v) / L
    dx_dt = v
    dv_dt = (Bl * i - k * x - b * v) / m
    return np.array([di_dt, dx_dt, dv_dt])

# Carregar o áudio de entrada
input_audio, sample_rate = sf.read('TC02-in.wav')

# Verificar se o áudio é estéreo (2D) e selecionar um canal
if input_audio.ndim > 1:
    input_audio = input_audio[:, 0]  # Use apenas o primeiro canal

# Diminui a duração do teste
max_duration = 10  # Usar apenas os primeiros 3 segundos para simulação
duration = min(len(input_audio) / sample_rate, max_duration)
time = np.linspace(0, duration, int(sample_rate * duration / 10))  # Reduz o número de pontos de avaliação

# Limitar o tamanho do input_audio ao tamanho de time
input_audio = input_audio[:len(time)]

# Simulação do modelo linear
y0 = [0, 0, 0]  # condições iniciais (i, x, v)
linear_solution = integrate.solve_ivp(linear_model, [0, duration], y0, t_eval=time, args=(input_audio,), method='RK45')
i_linear = linear_solution.y[0]  # corrente
x_linear = linear_solution.y[1]    # posição do cone (deslocamento)
v_linear = linear_solution.y[2]    # velocidade

# Aceleração
u_dot_linear = np.gradient(x_linear, time)
u_double_dot_linear = np.gradient(u_dot_linear, time)

# Análise de Fourier para Vin(t) e ü(t)
Vin_freq = fft.fft(input_audio[:len(time)])  # Limitar a FFT ao tamanho do tempo
u_double_dot_linear_freq = fft.fft(u_double_dot_linear)
freqs = fft.fftfreq(len(Vin_freq), 1 / sample_rate)

# Salvar a resposta do modelo linear como áudio
sf.write('TC02-out1Teste.wav', u_double_dot_linear[:len(time)], sample_rate)  # Limitar o tamanho do áudio de saída

# Plotar Vin(t) e Vin(jω)
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(time, input_audio, label="Vin(t)")
plt.title("Vin(t) - Sinal de Entrada no Tempo")
plt.xlabel("Tempo (s)")
plt.ylabel("Amplitude")
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(freqs, np.abs(Vin_freq), label="Vin(jω)")
plt.title("Vin(jω) - Sinal de Entrada na Frequência")
plt.xlabel("Frequência (Hz)")
plt.ylabel("Magnitude")
plt.xlim(-2000,2000)
plt.legend()
plt.tight_layout()
plt.show()

# Comparação das respostas do modelo linear
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(time, u_double_dot_linear, label="Aceleração ü(t)")
plt.title("Aceleração ü(t) - Modelo Linear")
plt.xlabel("Tempo (s)")
plt.ylabel("Aceleração")
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(freqs, np.abs(u_double_dot_linear_freq), label="ü(jω)")
plt.title("ü(jω) - Aceleração na Frequência")
plt.xlabel("Frequência (Hz)")
plt.xlim(-2000,2000)
plt.ylabel("Magnitude")
plt.legend()
plt.tight_layout()
plt.show()

# Gráfico da corrente i(t)
plt.subplot(3, 1, 1)
plt.plot(time, i_linear, label="i(t)", color='orange')
plt.title("Corrente i(t) - Resposta do Alto-Falante")
plt.xlabel("Tempo (s)")
plt.ylabel("Corrente (A)")
plt.legend()

# Gráfico da posição x(t)
plt.subplot(3, 1, 2)
plt.plot(time, x_linear, label="x(t)", color='blue')
plt.title("Posição x(t) - Resposta do Alto-Falante")
plt.xlabel("Tempo (s)")
plt.ylabel("Deslocamento (m)")
plt.legend()

# Gráfico da velocidade u̇(t)
plt.subplot(3, 1, 3)
plt.plot(time, u_dot_linear, label="u̇(t)", color='green')
plt.title("Velocidade u̇(t) - Resposta do Alto-Falante")
plt.xlabel("Tempo (s)")
plt.ylabel("Velocidade (m/s)")
plt.legend()


plt.tight_layout()
plt.show()
